# FPL data

Official FPL API (no auth). The API only ever serves the **current** season — right now that's **2025-26** (just finished), so we archive it before it rolls over to 2026-27.

In [1]:
import os, time, json
import requests
import pandas as pd
from requests.exceptions import HTTPError, RequestException

BASE_URL = "https://fantasy.premierleague.com/api/"

def get_element_summary(player_id):
    """Per-gameweek history for one player (retries on rate-limit)."""
    while True:
        try:
            r = requests.get(BASE_URL + f"element-summary/{player_id}/")
            r.raise_for_status()
            return r.json()
        except (RequestException, HTTPError):
            time.sleep(0.3)

In [2]:
season = "2025-26"
season_dir = os.path.join(os.path.abspath(".."), "data", season)
os.makedirs(season_dir, exist_ok=True)

In [3]:
# bootstrap-static: snapshot of season-to-date totals for every player, plus teams & events.
data = requests.get(BASE_URL + "bootstrap-static/").json()
fixtures = requests.get(BASE_URL + "fixtures/").json()  # canonical match list (each game once)

with open(os.path.join(season_dir, f"bootstrap_{season}.json"), "w") as f:
    json.dump(data, f)
pd.DataFrame(fixtures).to_csv(os.path.join(season_dir, f"fixtures_{season}.csv"), index=False)
pd.DataFrame(data["teams"]).to_csv(os.path.join(season_dir, f"teams_{season}.csv"), index=False)
pd.DataFrame(data["element_types"]).to_csv(os.path.join(season_dir, f"element_types_{season}.csv"), index=False)

print(f"{len(data['elements'])} players, {len(data['events'])} gameweeks, {len(fixtures)} fixtures "
      f"(GW1 {data['events'][0]['deadline_time'][:10]} -> GW38 {data['events'][-1]['deadline_time'][:10]})")

841 players, 38 gameweeks, 380 fixtures (GW1 2025-08-15 -> GW38 2026-05-24)


In [4]:
gw_path   = os.path.join(season_dir, f"merged_gw_{season}.csv")
past_path = os.path.join(os.path.dirname(season_dir), "history_past.csv")

# Lookup dicts are always needed by downstream cells.
pcode      = {p["id"]: p["code"]                    for p in data["elements"]}
name       = {p["id"]: p["web_name"]                for p in data["elements"]}
p_team     = {p["id"]: p["team"]                    for p in data["elements"]}
p_pos      = {p["id"]: p["element_type"]            for p in data["elements"]}
team_name  = {t["id"]: t["name"]                    for t in data["teams"]}
team_short = {t["id"]: t["short_name"]              for t in data["teams"]}
pos_short  = {e["id"]: e["singular_name_short"]     for e in data["element_types"]}

if os.path.exists(gw_path) and os.path.exists(past_path):
    print("Loading cached data...")
    gw   = pd.read_csv(gw_path)
    past = pd.read_csv(past_path)
    print(f"Loaded {len(gw)} GW rows, {len(past)} history_past rows")
else:
    player_ids = [p["id"] for p in data["elements"]]
    print(f"Fetching {len(player_ids)} players...")
    rows, past_rows = [], []
    for n, pid in enumerate(player_ids, 1):
        summary = get_element_summary(pid)
        rows.extend(summary["history"])
        past_rows.extend(summary["history_past"])
        if n % 200 == 0:
            print(f"  {n}/{len(player_ids)}")
        time.sleep(0.05)

    gw = pd.DataFrame(rows)
    gw["code"]                = gw["element"].map(pcode)
    gw["player_id"]           = gw["element"]
    gw["web_name"]            = gw["element"].map(name)
    gw["position"]            = gw["element"].map(p_pos).map(pos_short)
    gw["team"]                = gw["element"].map(p_team)
    gw["team_name"]           = gw["team"].map(team_name)
    gw["team_short"]          = gw["team"].map(team_short)
    gw["opponent_team_name"]  = gw["opponent_team"].map(team_name)
    gw["opponent_team_short"] = gw["opponent_team"].map(team_short)
    gw = gw.sort_values(["round", "element"]).reset_index(drop=True)
    gw.to_csv(gw_path, index=False)
    print(f"Saved {len(gw)} rows x {gw['round'].nunique()} GWs -> merged_gw_{season}.csv")

    past = pd.DataFrame(past_rows).rename(columns={"element_code": "code"})
    past.to_csv(past_path, index=False)
    print(f"Saved {len(past)} prior-season rows ({sorted(past['season_name'].unique())}) -> history_past.csv")


Loading cached data...
Loaded 29747 GW rows, 3187 history_past rows


In [5]:
# Sanity check: per-GW points must sum to each player's season total in the snapshot.
boot = {p["id"]: p["total_points"] for p in data["elements"]}
summed = gw.groupby("element")["total_points"].sum()
print("Season totals reconcile:", all(summed[e] == boot[e] for e in summed.index))

# Example: one player's season, gameweek by gameweek
gw[gw["web_name"] == "Raya"][["round", "team_name", "opponent_team_name", "minutes", "goals_scored", "assists", "total_points"]]

Season totals reconcile: True


,round,team_name,opponent_team_name,minutes,goals_scored,assists,total_points
0,1,Arsenal,Man Utd,90,0,0,10
690,2,Arsenal,Leeds,90,0,0,6
1395,3,Arsenal,Liverpool,90,0,0,2
2107,4,Arsenal,Nott'm Forest,90,0,0,6
2847,5,Arsenal,Man City,90,0,0,2
3588,6,Arsenal,Newcastle,90,0,0,2
4330,7,Arsenal,West Ham,90,0,0,6
5073,8,Arsenal,Fulham,90,0,0,6
5818,9,Arsenal,Crystal Palace,90,0,0,6
6564,10,Arsenal,Burnley,90,0,0,6


## Season totals (prior-season baselines)

Two sources, both keyed by the permanent **`code`**:
- **Current season** (`season_totals_{season}.csv`) — rich aggregates from the snapshot
  (points_per_game, now_cost, names, position, xG/xA, ...).
- **Prior seasons** (`season_totals_2024-25.csv`, ...) — derived from `history_past` above;
  fewer columns (pure season stats, no name/team/position) but lets us build a leak-free
  baseline, e.g. predict 25-26 GW1 from 24-25 totals.

In [8]:
# Current-season totals from the snapshot.
cols = ["code", "id", "web_name", "first_name", "second_name", "team", "element_type",
        "total_points", "points_per_game", "minutes", "starts", "now_cost",
        "goals_scored", "assists", "clean_sheets", "bonus", "bps",
        "expected_goals", "expected_assists", "expected_goal_involvements", "ict_index"]
totals = pd.DataFrame(data["elements"])[cols].copy()
totals["team_name"]  = totals["team"].map(team_name)
totals["team_short"] = totals["team"].map(team_short)
totals["position"]   = totals["element_type"].map(pos_short)
totals = totals.rename(columns={"id": f"element_{season.replace('-', '_')}"})
totals = totals.sort_values("total_points", ascending=False).reset_index(drop=True)
totals.to_csv(os.path.join(season_dir, f"season_totals_{season}.csv"), index=False)
print(f"{season}: {len(totals)} players, unique codes {totals['code'].nunique()}")

# Prior-season totals from history_past (e.g. 2024-25), saved per season folder.
prior = "2024/25"
prior_dir = os.path.join(os.path.dirname(season_dir), prior.replace("/", "-"))
os.makedirs(prior_dir, exist_ok=True)
prior_totals = past[past["season_name"] == prior].sort_values("total_points", ascending=False).reset_index(drop=True)
prior_totals["web_name"] = prior_totals["code"].map({p["code"]: p["web_name"] for p in data["elements"]})
prior_totals.to_csv(os.path.join(prior_dir, f"season_totals_{prior.replace('/', '-')}.csv"), index=False)
print(f"{prior}: {len(prior_totals)} players, unique codes {prior_totals['code'].nunique()}")

2025-26: 841 players, unique codes 841
2024/25: 534 players, unique codes 534
